# Window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-06-window-functions.ipynb)

A window function computes a value over a set of rows **related to the current row**, without collapsing
them. A `GROUP BY` returns one row per group. A window keeps every row and adds a column, so you can show an
order next to its customer's running total, or next to its rank.

The syntax is `function() OVER (PARTITION BY ... ORDER BY ... frame)`. `PARTITION BY` restarts the window for
each group, `ORDER BY` defines the sequence, and the frame says which earlier and later rows are included.

Previous: [joins](sql-05-joins.ipynb). Next: [deduplication](sql-07-deduplication.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## The example data

The `customers` and `orders` tables, plus a small `pay` table with tied salaries to make ranking differences visible.

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

In [ ]:
create_view(
    "pay",
    "name STRING, salary INT",
    [("Alice", 150), ("Bob", 120), ("Carol", 120), ("Dave", 95)],
)

## Ranking functions

`ROW_NUMBER` gives every row a distinct number, breaking ties arbitrarily unless you add a tie-breaker to
`ORDER BY`. `RANK` gives tied rows the same number and then skips ahead. `DENSE_RANK` gives tied rows the same
number without skipping. Use a named `WINDOW` clause when several functions share one definition.

In [ ]:
check(
    """
    SELECT name, salary,
           RANK()       OVER by_salary AS rnk,
           DENSE_RANK() OVER by_salary AS dense_rnk
    FROM pay
    WINDOW by_salary AS (ORDER BY salary DESC)
    ORDER BY salary DESC, name
    """,
    [("Alice", 150, 1, 1), ("Bob", 120, 2, 2), ("Carol", 120, 2, 2), ("Dave", 95, 4, 3)],
    "RANK skips after a tie, DENSE_RANK does not",
)

### `ROW_NUMBER` with a tie-breaker

"The top order per customer" needs exactly one row per customer, so use `ROW_NUMBER` and break ties with a
unique column such as `order_id`. Without the tie-breaker the chosen row can change between runs.

In [ ]:
check(
    """
    SELECT customer_id, order_id, amount,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
    FROM orders
    WHERE customer_id = 1
    ORDER BY rn
    """,
    [(1, 2, 200, 1), (1, 3, 150, 2), (1, 1, 100, 3)],
    "ROW_NUMBER within a customer",
)

## Frames: `ROWS` versus `RANGE`

When an ordered window sees rows with the same `ORDER BY` value (peers), the frame decides what is included.
`ROWS` counts physical rows, so peers get different running totals. `RANGE` includes all peers, so they share
one total. Make the frame explicit whenever ties are possible.

In [ ]:
check(
    """
    SELECT name, salary,
           SUM(salary) OVER (ORDER BY salary, name
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_total,
           SUM(salary) OVER (ORDER BY salary
                             RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_total
    FROM pay
    ORDER BY salary, name
    """,
    [
        ("Dave", 95, 95, 95),
        ("Bob", 120, 215, 335),
        ("Carol", 120, 335, 335),
        ("Alice", 150, 485, 485),
    ],
    "ROWS versus RANGE with tied salaries",
)

## Running totals per customer

A running total is `SUM(...) OVER (PARTITION BY ... ORDER BY ...)`. Filter *before* the window with `WHERE`,
so excluded rows never enter the sum.

In [ ]:
check(
    """
    SELECT customer_id, order_date, amount,
           SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
    FROM orders
    WHERE status = 'completed' AND customer_id IN (1, 2)
    ORDER BY customer_id, order_date
    """,
    [
        (1, date(2023, 1, 5), 100, 100),
        (1, date(2023, 2, 10), 200, 300),
        (1, date(2023, 3, 15), 150, 450),
        (2, date(2023, 1, 20), 300, 300),
        (2, date(2023, 3, 20), 400, 700),
    ],
    "running total per customer",
)

## `LAG` for period-over-period change

`LAG(x)` returns the previous row's value in the window's order. Aggregate to one row per period first, then
apply `LAG`. Applied to raw orders it would compare with the previous *order*, not the previous month.

In [ ]:
check(
    """
    WITH monthly AS (
        SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY date_format(order_date, 'yyyy-MM')
    )
    SELECT month, revenue,
           LAG(revenue) OVER (ORDER BY month) AS previous_revenue,
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 2) AS growth_pct
    FROM monthly
    ORDER BY month
    """,
    [
        ("2023-01", 1020, None, None),
        ("2023-02", 655, 1020, -35.78),
        ("2023-03", 1320, 655, 101.53),
    ],
    "month-over-month growth",
)

## Share of total

`SUM(SUM(amount)) OVER ()` nests a window over an aggregate: the inner `SUM` totals each customer, and the outer
window totals those totals across the whole result. It gives each customer's share without a second query.

In [ ]:
check(
    """
    SELECT customer_id,
           SUM(amount) AS total,
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct_of_total
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
    ORDER BY customer_id
    """,
    [
        (1, 450, 15.03), (2, 700, 23.37), (3, 300, 10.02),
        (4, 320, 10.68), (5, 950, 31.72), (6, 275, 9.18),
    ],
    "each customer's share of completed revenue",
)

## Filtering on a window result

A window cannot appear in `WHERE`, because windows are evaluated after `WHERE`. Spark SQL has no `QUALIFY`
clause (some engines do), so filter the window result in an outer query. Here the top completed order per
customer:

In [ ]:
check(
    """
    SELECT customer_id, order_id, amount
    FROM (
        SELECT customer_id, order_id, amount,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
        FROM orders
        WHERE status = 'completed'
    )
    WHERE rn = 1
    ORDER BY customer_id
    """,
    [(1, 2, 200), (2, 6, 400), (3, 8, 180), (4, 10, 320), (5, 11, 500), (6, 13, 275)],
    "top completed order per customer",
)

## Takeaways

- A window keeps every row; a `GROUP BY` collapses them. Use windows when you need both the detail and a total.
- Give `ROW_NUMBER` a unique tie-breaker. Make the frame explicit when peers exist.
- Filter on a window result in an outer query.
- Aggregate to the right grain before `LAG`, so "previous" means what you think.